## Plate with hole (Kirsch benchmark): Gauss + direct solve vs matrix-free WQ + iterative solve

Same geometry, loading, boundary conditions, and exact solution as
`12_plate_with_hole.ipynb` (quarter annulus, Kirsch 1898 exact traction on the outer arc,
inner arc traction-free, symmetry on the two straight edges) -- reused here as the
accuracy benchmark, not reinvented, since it already has a validated closed-form solution
and an established k-refinement convergence study to build on.

What's new here: for every point of a (degree, refinement) grid, this notebook runs the
problem through **two** pipelines and compares both accuracy (against the exact Kirsch
solution) and cost:
- **reference**: `PatchIntegrator.integrate_stiffness()` (Gauss) + `scipy.sparse.linalg.spsolve`
  (direct, sparse LU) -- exactly `12_plate_with_hole.ipynb`'s own pipeline;
- **matrix-free**: `WQMatrixFreeStiffness` (Phase C) + the iterative solver from
  `matrix_free_solver.py` -- `K` never assembled.

In [ ]:
# Pin every native thread pool (future's OpenMP -- both its explicit #pragma omp
# regions and Eigen's own internal OpenMP-parallel matrix products -- and numpy/scipy's
# BLAS backend) to a single thread, so the Gauss-vs-matrix-free wall-clock comparison
# below measures algorithmic cost, not how many cores happened to be used. MUST be set
# before numpy/scipy/yeti_iga are imported: read once, at first use, not per call.
import os
for _env in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_env] = "1"

import time
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse.linalg as spla

from yeti_iga.future.bspline import (
    BSpline, BSplineSurface, ControlPointManager,
    Patch, PRefiner, SubdivisionRefiner,
    GlobalDOFManager, PatchDOFManager,
    PatchIntegrator, Material, PlaneStress, IGABasis1D,
    Traction, ScalarLocalOperator, WQMatrixFreeStiffness,
)
from yeti_iga.future.matrix_free_solver import solve as mf_solve
from yeti_iga.future.fast_diagonalization import FastDiagonalizationPreconditioner
from yeti_iga.future.postprocessing.plotting import plot_patches_2d

## The Kirsch problem: geometry, loading, and exact solution

Identical to `12_plate_with_hole.ipynb`: infinite plate with a circular hole of radius
`Rin=1`, quarter annulus out to `Rex=2`, uniaxial far-field tension `Tx=1` approximated by
the exact Kirsch traction applied on the *finite* outer arc (`direction=1, side=1`); the
inner arc (`direction=1, side=0`) is traction-free (natural BC, nothing to impose);
symmetry on the two straight edges (`u_y=0` on `theta=0`, `u_x=0` on `theta=90 deg`).

In [ ]:
Rin, Rex = 1.0, 2.0
Tx = 1.0
E, nu = 1e3, 0.3


class KirschTraction(Traction):
    """Exact Kirsch traction at the outer arc of the quarter annulus."""

    def __init__(self, Rin, Tx=1.0):
        super().__init__()
        self.Rin = Rin
        self.Tx = Tx

    def evaluate(self, pt):
        x, y = pt[0], pt[1]
        r = np.sqrt(x**2 + y**2)
        theta = np.arctan2(y, x)
        R2 = (self.Rin / r) ** 2
        R4 = (self.Rin / r) ** 4
        g1 = (self.Tx / 2) * (
            2 * np.cos(theta)
            - R2 * (2 * np.cos(theta) + 3 * np.cos(3 * theta))
            + 3 * R4 * np.cos(3 * theta)
        )
        g2 = (3 * self.Tx / 2) * np.sin(3 * theta) * (R4 - R2)
        return np.array([g1, g2])


def kirsch_displacement(pt):
    """Exact Kirsch displacement (plane stress) at physical point pt=(x,y).

    Derived by integrating the Kirsch strain field (Timoshenko & Goodier, Theory of
    Elasticity, Sections 71-72).
    """
    x, y = pt[0], pt[1]
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    a2 = Rin**2
    u_x = Tx / (2 * E) * (2 * (r + 2 * a2 / r) * np.cos(theta)
                          + (1 + nu) * a2 / r * (1 - a2 / r**2) * np.cos(3 * theta))
    u_y = Tx / (2 * E) * (-2 * (nu * r + (1 - nu) * a2 / r) * np.sin(theta)
                          + (1 + nu) * a2 / r * (1 - a2 / r**2) * np.sin(3 * theta))
    return np.array([u_x, u_y])


# Sanity checks (same as 12_plate_with_hole.ipynb): symmetry BCs hold on the exact
# solution, and the far-field limit is the expected uniaxial strain.
u_xaxis = kirsch_displacement(np.array([1.5, 0.0]))
u_yaxis = kirsch_displacement(np.array([0.0, 1.5]))
print(f'exact u at (1.5, 0)  : {u_xaxis}  (u_y should be 0)')
print(f'exact u at (0,   1.5): {u_yaxis}  (u_x should be 0)')
pt_far = np.array([100.0, 0.0])
print(f'far-field u_x at (100,0): {kirsch_displacement(pt_far)[0]:.5f}  (expected {Tx*100/E:.5f})')

In [ ]:
# Same 6-control-point quarter-ring as build_quarter_ring_k() below -- shown here,
# geometry-only (no DOF manager needed for a plot), as part of the problem description.
w_corner = 1.0 / np.sqrt(2.0)
mgr_geom = ControlPointManager(dim=2)
for (x, y, w) in [
    (Rin, 0., 1.0), (Rin, Rin, w_corner), (0., Rin, 1.0),
    (Rex, 0., 1.0), (Rex, Rex, w_corner), (0., Rex, 1.0),
]:
    mgr_geom.add_point([x, y], w)
su_geom = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))  # angular, degree 2
sv_geom = BSpline(1, np.array([0., 0., 1., 1.]))            # radial,  degree 1
patch_geom = Patch(BSplineSurface(su_geom, sv_geom), mgr_geom, list(range(6)), [3, 2])

plot_patches_2d(patch_geom, show_control_points=True, show_weights=True,
                title=f'Kirsch quarter-ring geometry (Rin={Rin}, Rex={Rex}) -- initial mesh')


## Boundary conditions and error norm

Same conventions as `12_plate_with_hole.ipynb`: symmetry BCs via `fixed_dofs_symmetry`
below, and the same two error norms, used to check the FE solution $u_h$ against the
exact Kirsch displacement $u$ for each pipeline.

**L2 error**, the plain displacement mismatch:

$$\|u_h - u\|_{L^2}^2 = \int_\Omega |u_h(\xi) - u(x(\xi))|^2 \, d\Omega$$

**H1 semi-norm**, the gradient mismatch:

$$|u_h - u|_{H^1}^2 = \int_\Omega |\nabla u_h(\xi) - \nabla u(x(\xi))|^2 \, d\Omega$$

and what this notebook (and `12_plate_with_hole.ipynb`) actually reports as "the H1
error" is the **full** H1 norm, not the semi-norm alone:

$$\|u_h - u\|_{H^1} = \sqrt{\|u_h - u\|_{L^2}^2 + |u_h - u|_{H^1}^2}$$

Both are computed the same way, via `ScalarLocalOperator`: a Python subclass supplies the
*pointwise* squared-error integrand at one Gauss point
(`compute_scalar_integrand(R, dRdx, dRdy, physical_point, u_local)` below), and
`PatchIntegrator.integrate_scalar_operator()` drives the Gauss loop, the physical-space
gradients, and the NURBS rationalisation -- so `L2ErrorOperator`/`H1SemiNormErrorOperator`
only have to express the *algebra* of the norm, not the integration machinery around it.

One subtlety worth calling out: the exact solution's own gradient (`grad_exact_fn` below,
passed to `H1SemiNormErrorOperator`) is `kirsch_displacement_gradient`, the **closed-form**
derivative of `kirsch_displacement` (chain rule on `(r, theta)`) -- not a finite
difference. An earlier version used a central difference (`h=1e-6`) instead, on the
grounds that at this `h` it cost nothing measurable in accuracy; that stopped being true
once this notebook's degree sweep went high enough that the true H1 error drops below a
finite difference's own noise floor (~`eps^(2/3)` ~ 3e-11 at the scheme's optimal step,
a limit `h` tuning can't avoid), which would otherwise plateau the error curve instead of
showing the real convergence.

In [ ]:
class L2ErrorOperator(ScalarLocalOperator):
    """Integrand |u_h - u_ex|^2 for the L2 error norm."""

    def __init__(self, u_exact_fn):
        super().__init__()
        self.u_exact_fn = u_exact_fn

    def compute_scalar_integrand(self, R, dRdx, dRdy, physical_point, u_local):
        R = np.asarray(R)
        u_x = np.asarray(u_local)[0::2]
        u_y = np.asarray(u_local)[1::2]
        u_ex = self.u_exact_fn(physical_point)
        return (R @ u_x - u_ex[0])**2 + (R @ u_y - u_ex[1])**2


def kirsch_displacement_gradient(pt):
    """Analytic gradient of kirsch_displacement, via the chain rule on (r, theta).

    u_x = C*(A(r) cos(theta) + B(r) cos(3 theta))
    u_y = C*(D(r) sin(theta) + B(r) sin(3 theta))
    with C = Tx/(2E), a2 = Rin**2,
    A(r) = 2r + 4 a2/r, D(r) = -2 nu r - 2(1-nu) a2/r,
    B(r) = (1+nu)(a2/r - a2**2/r**3).
    """
    x, y = pt[0], pt[1]
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    a2 = Rin**2
    C = Tx / (2 * E)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    cos_3t, sin_3t = np.cos(3 * theta), np.sin(3 * theta)

    A = 2 * r + 4 * a2 / r
    Ap = 2 - 4 * a2 / r**2
    B = (1 + nu) * (a2 / r - a2**2 / r**3)
    Bp = (1 + nu) * (3 * a2**2 / r**4 - a2 / r**2)
    D = -2 * nu * r - 2 * (1 - nu) * a2 / r
    Dp = -2 * nu + 2 * (1 - nu) * a2 / r**2

    dux_dr = C * (Ap * cos_t + Bp * cos_3t)
    dux_dth = C * (-A * sin_t - 3 * B * sin_3t)
    duy_dr = C * (Dp * sin_t + Bp * sin_3t)
    duy_dth = C * (D * cos_t + 3 * B * cos_3t)

    # chain rule: df/dx = f_r cos(theta) - f_theta sin(theta)/r
    #             df/dy = f_r sin(theta) + f_theta cos(theta)/r
    dux_dx = dux_dr * cos_t - dux_dth * sin_t / r
    dux_dy = dux_dr * sin_t + dux_dth * cos_t / r
    duy_dx = duy_dr * cos_t - duy_dth * sin_t / r
    duy_dy = duy_dr * sin_t + duy_dth * cos_t / r

    return np.array([[dux_dx, dux_dy], [duy_dx, duy_dy]])  # [[du1/dx, du1/dy],[du2/dx,du2/dy]]


class H1SemiNormErrorOperator(ScalarLocalOperator):
    """Integrand |grad(u_h) - grad(u_ex)|^2 for the H1 semi-norm -- same convention
    as 12_plate_with_hole.ipynb: full H1 norm = sqrt(L2_error^2 + H1_seminorm_error^2)."""

    def __init__(self, grad_exact_fn):
        super().__init__()
        self.grad_exact_fn = grad_exact_fn

    def compute_scalar_integrand(self, R, dRdx, dRdy, physical_point, u_local):
        dRdx = np.asarray(dRdx)
        dRdy = np.asarray(dRdy)
        u_x = np.asarray(u_local)[0::2]
        u_y = np.asarray(u_local)[1::2]
        grad_ex = self.grad_exact_fn(physical_point)
        return (
            (dRdx @ u_x - grad_ex[0, 0])**2 +
            (dRdy @ u_x - grad_ex[0, 1])**2 +
            (dRdx @ u_y - grad_ex[1, 0])**2 +
            (dRdy @ u_y - grad_ex[1, 1])**2
        )


def fixed_dofs_symmetry(patch, pdm):
    fixed = set()
    for cp in patch.boundary_control_points(0, 0):     # theta=0 edge (x-axis): block u_y
        fixed.add(pdm.get_global_dof_indices(cp)[1])
    for cp in patch.boundary_control_points(0, 1):      # theta=90deg edge (y-axis): block u_x
        fixed.add(pdm.get_global_dof_indices(cp)[0])
    return np.array(sorted(fixed))

## k-refinement geometry builder

Exactly `12_plate_with_hole.ipynb`'s convergence-study builder: canonical 6-control-point
quarter-ring (degree 2 angular / degree 1 radial), elevate both directions to the target
degree, then uniformly subdivide `n_levels` times.

In [ ]:
def build_quarter_ring_k(degree, n_levels):
    mgr = ControlPointManager(dim=2)
    w_c = 1.0 / np.sqrt(2.0)
    for (x, y, w) in [
        (Rin, 0., 1.0), (Rin, Rin, w_c), (0., Rin, 1.0),
        (Rex, 0., 1.0), (Rex, Rex, w_c), (0., Rex, 1.0),
    ]:
        mgr.add_point([x, y], w)

    su = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
    sv = BSpline(1, np.array([0., 0., 1., 1.]))
    patch = Patch(BSplineSurface(su, sv), mgr, list(range(6)), [3, 2])

    if degree > 2:
        PRefiner(direction=0, n_elevations=degree - 2).refine(patch)
    PRefiner(direction=1, n_elevations=degree - 1).refine(patch)
    SubdivisionRefiner(direction=0, n_levels=n_levels).refine(patch)
    SubdivisionRefiner(direction=1, n_levels=n_levels).refine(patch)

    n_cp = patch.n_cp
    gdm = GlobalDOFManager([2] * n_cp)
    pdm = PatchDOFManager(2, list(range(n_cp)), gdm)
    pk = Patch(patch.tensor, patch.cp_manager, list(patch.global_indices), list(patch.local_shape), pdm)

    su_k, sv_k = pk.tensor.components
    basis_u = IGABasis1D.build(su_k, su_k.degree + 1)
    basis_v = IGABasis1D.build(sv_k, sv_k.degree + 1)
    law = PlaneStress(Material(E=E, nu=nu))
    return pk, pdm, basis_u, basis_v, law

## The two pipelines

**Reference**: `integrate_stiffness()` (Gauss) + `scipy.sparse.linalg.spsolve` (direct,
sparse LU) on the explicitly sliced free-free system -- `12_plate_with_hole.ipynb`'s own
pipeline, unchanged.

**Matrix-free**: `WQMatrixFreeStiffness` + `matrix_free_solver.solve`, GMRES
preconditioned with `FastDiagonalizationPreconditioner` (same combination used in
`17_future_vs_pymfiga_matrix_free.ipynb`). GMRES rather than CG because, on curved
geometry, `WQMatrixFreeStiffness.apply()` is only *approximately* symmetric (weighted
quadrature's independently-fit gather/scatter matrices don't form an exact adjoint pair
once a spatially-varying material tensor sits between them -- see
`matrix_free_solver.py`'s module docstring) -- harmless on well-refined meshes, but it
can make plain CG fail to converge on the coarsest, most strongly curved
configurations. Preconditioned GMRES sidesteps that entirely, and cuts the iteration
count substantially in the process.

The load vector `F` is assembled via `integrate_boundary_load()` (Gauss) for **both**
pipelines -- matrix-free load assembly is out of this session's scope (see
`15_matrix_free_2d.ipynb`'s *Next steps*); only the stiffness action itself is
matrix-free here.


In [ ]:
def solve_gauss_direct(basis_u, basis_v, patch, law, F, free):
    integrator = PatchIntegrator(patch, basis_u, basis_v, law)
    t0 = time.perf_counter()
    K = integrator.integrate_stiffness()
    t_assemble = time.perf_counter() - t0

    K_ff = K.tocsr()[free][:, free].tocsc()
    t0 = time.perf_counter()
    u = np.zeros(F.shape[0])
    u[free] = spla.spsolve(K_ff, F[free])
    t_solve = time.perf_counter() - t0

    return u, t_assemble, t_solve


def solve_matrix_free_iterative(patch, pdm, law, F, fixed, ndof):
    t0 = time.perf_counter()
    wq_stiff = WQMatrixFreeStiffness(patch, law)
    # law= enables the thesis's geometry+material-aware FD correction (Eq. 2.15) --
    # without it (the "classic FD" default) this quarter-ring needs ~2x more GMRES
    # iterations. restart=200 (vs scipy's default 20) avoids the ~10% extra
    # iterations restarted GMRES pays on this preconditioned system (see
    # fast_diagonalization.py's docstring).
    precond = FastDiagonalizationPreconditioner(patch, pdm, n_dofs_per_cp=2, fixed_dofs=fixed, law=law)
    t_setup = time.perf_counter() - t0

    n_iter = [0]
    def _count(_xk):
        n_iter[0] += 1

    t0 = time.perf_counter()
    u, info = mf_solve(wq_stiff.apply, ndof, F, fixed, rtol=1e-10, maxiter=3000,
                       method='gmres', precondition_fn=precond.apply, callback=_count,
                       restart=200)
    t_solve = time.perf_counter() - t0

    return u, t_setup, t_solve, n_iter[0], info

## The sweep: 3 degrees x 7 refinement levels

Same grid shape as `12_plate_with_hole.ipynb`'s own convergence study (there: degrees
2-5, up to 32 elements/direction); extended here to 7 refinement levels (up to 64
elements/direction) to see the *asymptotic* convergence rate clearly, not just early
mesh sizes. The full H1 norm (`sqrt(L2^2 + H1-seminorm^2)`, same convention as
`12_plate_with_hole.ipynb` and the thesis's own Section 2.4.3, Fig. 2.9) is tracked for
both pipelines.


In [ ]:
degrees = [2, 3, 4]
levels = [0, 1, 2, 3, 4, 5, 6]

results = []
for degree in degrees:
    for n_levels in levels:
        patch, pdm, basis_u, basis_v, law = build_quarter_ring_k(degree, n_levels)
        integrator = PatchIntegrator(patch, basis_u, basis_v, law)
        ndof = 2 * patch.n_cp
        fixed = fixed_dofs_symmetry(patch, pdm)
        free = np.setdiff1d(np.arange(ndof), fixed)
        F = integrator.integrate_boundary_load(1, 1, KirschTraction(Rin, Tx))  # outer arc

        u_g, t_asm, t_solve_g = solve_gauss_direct(basis_u, basis_v, patch, law, F, free)
        u_mf, t_setup_mf, t_solve_mf, n_iter, info = solve_matrix_free_iterative(patch, pdm, law, F, fixed, ndof)

        errL2_g_sq = integrator.integrate_scalar_operator(L2ErrorOperator(kirsch_displacement), u_g)
        errL2_mf_sq = integrator.integrate_scalar_operator(L2ErrorOperator(kirsch_displacement), u_mf)
        errH1s_g_sq = integrator.integrate_scalar_operator(H1SemiNormErrorOperator(kirsch_displacement_gradient), u_g)
        errH1s_mf_sq = integrator.integrate_scalar_operator(H1SemiNormErrorOperator(kirsch_displacement_gradient), u_mf)

        err_g = np.sqrt(errL2_g_sq)
        err_mf = np.sqrt(errL2_mf_sq)
        errH1_g = np.sqrt(errL2_g_sq + errH1s_g_sq)
        errH1_mf = np.sqrt(errL2_mf_sq + errH1s_mf_sq)

        results.append(dict(
            degree=degree, n_levels=n_levels, ndof=ndof, nbel=2**n_levels,
            t_gauss=t_asm + t_solve_g, t_mf=t_setup_mf + t_solve_mf,
            err_gauss=err_g, err_mf=err_mf,
            errH1_gauss=errH1_g, errH1_mf=errH1_mf,
            n_iter=n_iter, info=info,
        ))
        print(f'p={degree} lvl={n_levels} ndof={ndof:5d}  '
              f'H1: Gauss={errH1_g:.2e} MF={errH1_mf:.2e}   '
              f'time: Gauss={t_asm+t_solve_g:7.3f}s  MF={t_setup_mf+t_solve_mf:7.3f}s   '
              f'(gmres, {n_iter:3d} iters{"" if info == 0 else f", info={info}!"})')


## Accuracy: convergence against the exact Kirsch solution

Gauss converges at the expected spectral (exponential-in-degree) IGA rate. Matrix-free
tracks it closely in the H1 norm at every degree and refinement level tested -- matching
the thesis's own Section 2.4.3 (Fig. 2.9) on this same benchmark.


In [ ]:
print(f'{"p":>2} {"lvl":>3} {"nbel":>4}  {"H1 order (Gauss/MF)":>22}')
for degree in degrees:
    rows = sorted([r for r in results if r['degree'] == degree], key=lambda r: r['n_levels'])
    for i in range(1, len(rows)):
        r0, r1 = rows[i - 1], rows[i]
        oH1_g = np.log2(r0['errH1_gauss'] / r1['errH1_gauss'])
        oH1_mf = np.log2(r0['errH1_mf'] / r1['errH1_mf'])
        print(f'{degree:2d} {r1["n_levels"]:3d} {r1["nbel"]:4d}  '
              f'{oH1_g:8.2f} / {oH1_mf:8.2f}')


The table below fits the apparent convergence order between each pair of consecutive
refinement levels (log2 of the error ratio). Gauss's order climbs to the theoretical
optimum (~degree) as expected; matrix-free's order tracks it closely at every level and
every degree -- no meaningful gap.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, degree in enumerate(degrees):
    rows = sorted([r for r in results if r['degree'] == degree], key=lambda r: r['n_levels'])
    nbels = [r['nbel'] for r in rows]
    errH1_g = [r['errH1_gauss'] for r in rows]
    errH1_mf = [r['errH1_mf'] for r in rows]
    ax.loglog(nbels, errH1_g, 'o-', color=colors[i], label=f'degree {degree}, Gauss + direct')
    ax.loglog(nbels, errH1_mf, 's--', color=colors[i], alpha=0.7, label=f'degree {degree}, matrix-free + iterative')

ax.set_xlabel('Number of elements by direction')
ax.set_ylabel('H1 error')
ax.set_title('Plate with hole: H1 error, Gauss+direct vs matrix-free+iterative\n'
             '(same axes convention as the thesis, Section 2.4.3, Fig. 2.9)')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


**The matrix-free curve tracks Gauss almost exactly in the H1 norm, at every degree
tested** -- no visible gap, matching the thesis's own Fig. 2.9 (also H1, also this
Kirsch benchmark).


## Cost: total wall time, setup/assembly + solve

**Every native thread pool is pinned to 1 thread** (see the imports cell) -- `future`'s
OpenMP (both its explicit `#pragma omp` regions and Eigen's own internal
OpenMP-parallelized matrix products) and numpy/scipy's BLAS backend both default to
using every core otherwise, which would silently turn a wall-clock comparison into "who
uses more cores" rather than "which algorithm is cheaper". This matters here more than
expected -- see below.

Unlike the earlier (unpreconditioned CG/GMRES) version of this notebook -- where Gauss +
direct solve won everywhere -- **preconditioned GMRES changes the picture**, and doing so
with the thesis's actual geometry+material-aware FD correction (`law=` passed to
`FastDiagonalizationPreconditioner`, Eq. (2.15) -- see `fast_diagonalization.py`'s
docstring; an earlier version of this notebook used the "classic", geometry-blind FD
variant and needed roughly 2x more GMRES iterations) changes it further still: the
iteration count grows over the first few refinement levels, then **plateaus around
~40-50** (not perfectly flat, but far closer to the thesis's own reported ~39 -- Table
2.3, Fig. 2.10 -- than the classic variant's ~100-115 plateau) regardless of degree.
Gauss's direct solve, meanwhile, still pays for a sparse LU factorization whose cost
grows faster than matrix-free's near-constant-iteration cost as `ndof` grows -- so a
genuine **crossover appears**, and single-threading makes it *wider*, not narrower:

- degree 4, finest mesh tested (`ndof=9248`, 43 iterations): Gauss `4.17s` vs
  matrix-free `0.75s` -- over **5.5x faster**. Multi-threaded, this was `3.58s` vs
  `0.77s` (~4.6x): Gauss's own time grew by ~16% under single-threading while
  matrix-free's stayed essentially flat -- meaning Gauss's direct solve was quietly
  benefiting from multiple cores (sparse-LU factorization routines commonly are
  multi-threaded internally) while `future`'s own OpenMP parallelism wasn't actually
  buying much at this problem size to begin with. The crossover itself shows up early
  too -- already around `ndof=288` (level 3), not just at the finest mesh.
- degree 3, finest mesh tested (`ndof=8978`, 43 iterations): Gauss `1.66s` vs
  matrix-free `0.67s` -- **2.5x faster** (multi-threaded: `1.53s`/`0.64s`, ~2.4x --
  little changed here).
- degree 2, finest mesh tested (`ndof=8712`, 42 iterations): Gauss `0.48s` vs
  matrix-free `0.49s` -- essentially **at parity** now (multi-threaded: `0.44s`/`0.63s`,
  matrix-free noticeably behind) -- single-threading closed this gap too.

The crossover point clearly moves to smaller `ndof` as degree increases -- consistent
with Gauss's direct-solve cost scaling worse with problem size/bandwidth than
matrix-free's near-constant-iteration-count cost under this preconditioner.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for i, degree in enumerate(degrees):
    rows = [r for r in results if r['degree'] == degree]
    ndofs = [r['ndof'] for r in rows]
    t_g = [r['t_gauss'] for r in rows]
    t_mf = [r['t_mf'] for r in rows]
    ax.loglog(ndofs, t_g, 'o-', color=colors[i], label=f'degree {degree}, Gauss + direct')
    ax.loglog(ndofs, t_mf, 's--', color=colors[i], alpha=0.7, label=f'degree {degree}, matrix-free + iterative')

ax.set_xlabel('number of DOFs')
ax.set_ylabel('total wall time (s): assembly/setup + solve')
ax.set_title('Plate with hole: cost, Gauss+direct vs matrix-free+iterative')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, which='both', alpha=0.3)
plt.show()

## Summary

- **Both pipelines converge to the exact Kirsch solution** -- confirming
  `WQMatrixFreeStiffness` + the new iterative solver produce physically correct results
  on this validated benchmark, not just internally-consistent ones.
- **Matrix-free tracks Gauss closely in the H1 norm**, at every degree and every
  refinement level tested (up to 64 elements/direction) -- no visible gap, matching what
  the thesis's own Section 2.4.3 (Fig. 2.9) shows for this same benchmark.
- **The preconditioner must include geometry and material information to be
  competitive.** The "classic" fast-diagonalization method (built purely from the
  parametric 1D mass/stiffness pencils, no knowledge of the curved geometry or the
  material) needs roughly 2x more GMRES iterations than the thesis's own corrected
  variant (Eq. (2.15), now available via `law=` on `FastDiagonalizationPreconditioner`)
  -- ~100-115 iterations versus ~40-50, the latter close to the thesis's own reported
  ~39 (Table 2.3, Fig. 2.10). This was the actual, correctable cause of an earlier
  version of this notebook's weaker-than-expected crossover.
- **With the corrected preconditioner AND a fair single-thread comparison, matrix-free
  wins by a clear margin at degree >= 3, and reaches parity even at degree 2.** Degree
  4 at `ndof=9248`: matrix-free is **5.5x faster** than Gauss + direct; degree 3 at
  `ndof=8978`: **2.5x faster**; degree 2 at `ndof=8712`: essentially tied. All three
  margins are *wider* single-threaded than they were multi-threaded (previously 4.6x /
  2.4x / matrix-free clearly behind) -- pinning every thread pool to 1 (see the imports
  cell) revealed that part of Gauss's apparent speed came from its direct solver using
  multiple cores, not from a fundamentally cheaper algorithm at these sizes. The
  crossover point moves to smaller problems as degree increases, since Gauss's
  sparse-LU factorization cost keeps growing with `ndof` while the preconditioned
  iteration count stays roughly flat.

None of this makes the matrix-free pipeline strictly better or worse -- it depends on
degree, problem size, and (as found here) getting the preconditioner right and the
comparison fair. Matrix-free's per-application cost still isn't fused (see
`15_matrix_free_2d.ipynb`'s *Next steps*), so today's crossover point is not a hard
limit. The motivating regime -- very large or 3D problems where assembling and
factorizing `K` is itself prohibitive -- is not what this 2D benchmark tests, but the
crossover already visible here at modest sizes is a first concrete sign of that
motivation showing up in practice.